# LoopedQwen — Experiment 003: loop-state noise

Ноутбук принимает ZIP с кодовой базой, готовит FineWeb, обучает контроль и две noise-абляции, выполняет evaluation и скачивает компактный ZIP для последующего анализа. Перед запуском выберите **Runtime → Change runtime type → T4 GPU** (или более мощную GPU).

In [ ]:
# 1. Загрузите .zip с репозиторием LoopedQwen
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Ожидался ровно один ZIP, получено: {zip_names}')

extract_dir = Path('/content/loopedqwen_source')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with ZipFile(zip_names[0]) as archive:
    archive.extractall(extract_dir)

projects = list(extract_dir.rglob('pyproject.toml'))
if len(projects) != 1:
    raise RuntimeError(f'Не удалось однозначно найти корень проекта: {projects}')
repo_root = projects[0].parent
os.chdir(repo_root)
print('Repository root:', repo_root)
print('Experiment files:', sorted(str(p.relative_to(repo_root)) for p in (repo_root / 'experiments/003_loop_state_noise').rglob('*') if p.is_file()))

In [ ]:
# 2. Установка и проверка GPU
%pip install -q -e .

import subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU не обнаружена. Включите GPU runtime и перезапустите ноутбук.')
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
runtime = subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout
Path('runtime_info.txt').write_text(runtime, encoding='utf-8')
print(runtime)

In [ ]:
# 3. Настройки запуска
VARIANTS = ['control_r16', 'relative_s003_r16', 'spherical_s003_r16']
TRAIN_TOKENS = 10_000_000
VAL_TOKENS = 1_000_000
TOKENIZER_DOCUMENTS = 100_000
EVAL_BATCHES = 50
RESUME_IF_POSSIBLE = True

# Для быстрой технической проверки можно временно оставить один вариант
# и уменьшить max_train_tokens в соответствующем YAML. Для финального
# сравнения не меняйте seed, данные, tokenizer и budget между вариантами.
print('Variants:', VARIANTS)

In [ ]:
# 4. Tokenizer и фиксированный FineWeb subset (пропускается, если уже есть в ZIP)
import subprocess

def run(command):
    print('>', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=repo_root, check=True)

if not Path('tokenizer/tokenizer.json').is_file():
    run([sys.executable, 'scripts/train_tokenizer.py', '--output-dir', 'tokenizer',
         '--vocab-size', '16000', '--documents', TOKENIZER_DOCUMENTS])
else:
    print('Tokenizer найден — обучение пропущено')

need_data = not Path('data/train.bin').is_file() or not Path('data/val.bin').is_file()
if need_data:
    run([sys.executable, 'scripts/prepare_data.py', '--tokenizer', 'tokenizer',
         '--output-dir', 'data', '--train-tokens', TRAIN_TOKENS, '--val-tokens', VAL_TOKENS])
else:
    print('Token data найдены — подготовка пропущена')

In [ ]:
# 5. До дорогого запуска проверяем модель, round-trip и noise-инварианты
run([sys.executable, '-m', 'pytest', '-q'])
run([sys.executable, 'scripts/sanity_check.py'])

In [ ]:
# 6. Обучение и evaluation. Ячейку безопасно запускать повторно с RESUME_IF_POSSIBLE=True.
for variant in VARIANTS:
    command = [sys.executable, 'experiments/003_loop_state_noise/run.py',
               '--variant', variant, '--eval-batches', EVAL_BATCHES]
    if RESUME_IF_POSSIBLE:
        command.append('--resume')
    run(command)

In [ ]:
# 7. Итоговая таблица
run([sys.executable, 'experiments/003_loop_state_noise/summarize.py'])
import pandas as pd
summary = pd.read_csv('experiments/003_loop_state_noise/results/summary.csv')
display(summary)

In [ ]:
# 8. Создание и скачивание ZIP для анализа
RESULT_ZIP = Path('/content/experiment_003_results.zip')
run([sys.executable, 'experiments/003_loop_state_noise/collect_results.py',
     '--output', RESULT_ZIP])
print(f'Готово: {RESULT_ZIP} ({RESULT_ZIP.stat().st_size / 1_000_000:.1f} MB)')
files.download(str(RESULT_ZIP))

После скачивания отправьте `experiment_003_results.zip` для анализа. Веса в архив не входят; для публикации лучшего checkpoint используйте `scripts/upload_to_hub.py` после выбора победителя.